In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# -----------------------------
# 1. Load Dataset
# -----------------------------
df = pd.read_csv("final_integrated_model1.csv")

# -----------------------------
# 2. Define Target Variable
# -----------------------------
y = df["TOTAL_TRAFFIC"]

# -----------------------------
# 3. Drop Ignored Columns
# -----------------------------
X = df.drop(columns=[
    "TOTAL_TRAFFIC",
    "Month",
    "Month_Year",
    "Facility_Order"
])

# -----------------------------
# 4. Identify Categorical Columns
# -----------------------------
cat_cols = ["Facility", "Direction"]
num_cols = [c for c in X.columns if c not in cat_cols]

# -----------------------------
# 5. Preprocessing Pipeline
# -----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

# -----------------------------
# 6. Gradient Boosting Model
# (Matched to H2O settings)
# -----------------------------
model = GradientBoostingRegressor(
    n_estimators=108,       # ntrees
    max_depth=12,           # max_depth
    min_samples_leaf=1,     # min_rows
    subsample=0.6,          # sample_rate
    random_state=42
)

# -----------------------------
# 7. Full Pipeline
# -----------------------------
pipeline = Pipeline([
    ("pre", preprocessor),
    ("model", model)
])

# -----------------------------
# 8. 5-Fold Cross Validation
# -----------------------------
kf = KFold(n_splits=5, shuffle=False)

rmses = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    rmses.append(rmse)

print("Cross-Validated RMSE:", np.mean(rmses))

Cross-Validated RMSE: 194282.0217206072


In [2]:
!pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------- ----------- 1.0/1.5 MB 12.4 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 6.0 MB/s  0:00:00


In [5]:
print(df["TOTAL_TRAFFIC"].describe())

count    1.920000e+02
mean     1.001329e+07
std      6.294216e+05
min      8.545155e+06
25%      9.797835e+06
50%      1.009193e+07
75%      1.057833e+07
max      1.069256e+07
Name: TOTAL_TRAFFIC, dtype: float64


In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# -----------------------------
# 1) Load dataset
# -----------------------------
df = pd.read_csv("final_integrated_model1.csv")

# -----------------------------
# 2) Target
# -----------------------------
y = df["TOTAL_TRAFFIC"]

# -----------------------------
# 3) Build features (NO leakage)
#    Drop:
#      - target
#      - ignored columns from H2O
#      - leakage variables (payment + violations)
# -----------------------------
drop_cols = [
    "TOTAL_TRAFFIC",
    "Month", "Month_Year", "Facility_Order",
    "TOTAL_EZPASS", "TOTAL_CASH", "TOTAL_VIOLATIONS"
]

X = df.drop(columns=[c for c in drop_cols if c in df.columns])

# -----------------------------
# 4) Categorical vs numeric columns
# -----------------------------
cat_cols = [c for c in ["Facility", "Direction"] if c in X.columns]
num_cols = [c for c in X.columns if c not in cat_cols]

# -----------------------------
# 5) Preprocess: OneHot for categoricals
# -----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

# -----------------------------
# 6) Model (GBM)
#    Keep similar structure to your H2O GBM run
# -----------------------------
model = GradientBoostingRegressor(
    n_estimators=108,
    max_depth=12,
    min_samples_leaf=1,
    subsample=0.6,
    random_state=42
)

pipeline = Pipeline([
    ("pre", preprocessor),
    ("model", model)
])

# -----------------------------
# 7) 5-fold CV RMSE
#    (manual RMSE for older sklearn)
# -----------------------------
kf = KFold(n_splits=5, shuffle=False)

rmses = []
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    rmses.append(rmse)

print("No-leakage CV RMSE:", np.mean(rmses))
print("No-leakage CV RMSE (% of mean traffic):",
      (np.mean(rmses) / y.mean()) * 100)

No-leakage CV RMSE: 315453.32788895676
No-leakage CV RMSE (% of mean traffic): 3.1503477662267056
